In [2]:
import pandas as pd 

df = pd.read_csv("data/interim/gdelt_event_context_daily/2026/02/22/20260222_events_full.csv")

df.head()

print("df shape:", df.shape)
print("non-null sourceurl:", df["sourceurl"].notna().sum())
print("sample sourceurl:", df["sourceurl"].dropna().head(3).tolist())

df shape: (65302, 62)
non-null sourceurl: 65302
sample sourceurl: ['https://www.bostonglobe.com/2026/02/21/world/gisele-pelicot-speaks-out-abuse/', 'https://www.bostonglobe.com/2026/02/21/world/gisele-pelicot-speaks-out-abuse/', 'https://www.bostonglobe.com/2026/02/21/world/gisele-pelicot-speaks-out-abuse/']


In [3]:
print("Total rows:", len(df))
print("Total columns:", df.shape[1])

Total rows: 65302
Total columns: 62


In [4]:
na_counts = df.isna().sum()
na_percent = (na_counts / len(df)) * 100

na_summary = pd.DataFrame({
    "non_null_count": df.notna().sum(),
    "na_count": na_counts,
    "na_percent": na_percent.round(2)
}).sort_values("na_count", ascending=False)

na_summary

,non_null_count,na_count,na_percent
actor2type3code,25,65277,99.96
actor1type3code,47,65255,99.93
actor2religion2code,136,65166,99.79
actor1religion2code,183,65119,99.72
actor2ethniccode,281,65021,99.57
...,...,...,...
numentions,65302,0,0.00
numsources,65302,0,0.00
numarticles,65302,0,0.00
avgtone,65302,0,0.00


In [5]:
unique_counts = df.nunique().sort_values(ascending=False)
unique_counts

globaleventid          65302
sourceurl              12655
avgtone                 9247
actiongeo_fullname      4095
actiongeo_featureid     4011
                       ...  
quadclass                  4
year                       3
isrootevent                2
dateadded                  1
ingest_time                1
Length: 62, dtype: int64

In [6]:
ZAMBIA_GEO_FIPS = "ZA"
ZAMBIA_ACTOR_CAMEO = "ZMB"

TEXT_TERMS = [
    "zambia", "zambian",
    "lusaka", "copperbelt", "ndola", "kitwe", "kabwe", "livingstone",
    "chingola", "solwezi", "chipata", "kasama", "mongu"
]

def contains_any(series, terms):
    s = series.fillna("").str.lower()
    mask = False
    for t in terms:
        mask = mask | s.str.contains(t, regex=False)
    return mask

geo_country_cols = ["actiongeo_countrycode", "actor1geo_countrycode", "actor2geo_countrycode"]
geo_adm1_cols    = ["actiongeo_adm1code", "actor1geo_adm1code", "actor2geo_adm1code"]
geo_full_cols    = ["actiongeo_fullname", "actor1geo_fullname", "actor2geo_fullname"]
actor_country_cols = ["actor1countrycode", "actor2countrycode"]
actor_name_cols = ["actor1name", "actor2name"]

geo_country_hit = False
for c in geo_country_cols:
    if c in df.columns:
        geo_country_hit = geo_country_hit | (df[c] == ZAMBIA_GEO_FIPS)

geo_adm1_hit = False
for c in geo_adm1_cols:
    if c in df.columns:
        geo_adm1_hit = geo_adm1_hit | df[c].fillna("").str.startswith(ZAMBIA_GEO_FIPS)

actor_country_hit = False
for c in actor_country_cols:
    if c in df.columns:
        actor_country_hit = actor_country_hit | (df[c] == ZAMBIA_ACTOR_CAMEO)

geo_text_hit = False
for c in geo_full_cols:
    if c in df.columns:
        geo_text_hit = geo_text_hit | contains_any(df[c], TEXT_TERMS)

actor_text_hit = False
for c in actor_name_cols:
    if c in df.columns:
        actor_text_hit = actor_text_hit | contains_any(df[c], ["zambia", "zambian"])

zambia_mask = geo_country_hit | geo_adm1_hit | actor_country_hit | geo_text_hit | actor_text_hit
df_zambia = df[zambia_mask].copy()

In [7]:
import pandas as pd
import numpy as np

def summarise_zambia(df_zambia: pd.DataFrame):
    print("="*80)
    print("ZAMBIA DATASET SUMMARY")
    print("="*80)

    # ---------------------------------------------------------
    # 1️⃣ BASIC INFO
    # ---------------------------------------------------------
    print("\n--- BASIC INFO ---")
    print(f"Total rows: {len(df_zambia):,}")
    print(f"Unique GlobalEventID: {df_zambia['globaleventid'].nunique():,}")

    if "sqldate" in df_zambia.columns:
        df_zambia["sqldate"] = pd.to_datetime(df_zambia["sqldate"], format="%Y%m%d", errors="coerce")
        print(f"Date range: {df_zambia['sqldate'].min()} → {df_zambia['sqldate'].max()}")


    # ---------------------------------------------------------
    # 3️⃣ EVENT TYPE DISTRIBUTION
    # ---------------------------------------------------------
    print("\n--- EVENT ROOT CODE DISTRIBUTION ---")
    if "eventrootcode" in df_zambia.columns:
        print(df_zambia["eventrootcode"].value_counts().head(15))

    print("\n--- QUADCLASS DISTRIBUTION ---")
    if "quadclass" in df_zambia.columns:
        print(df_zambia["quadclass"].value_counts())

    print("\n--- GOLDSTEIN SCALE SUMMARY ---")
    if "goldsteinscale" in df_zambia.columns:
        print(df_zambia["goldsteinscale"].describe())

    # ---------------------------------------------------------
    # 4️⃣ TONE & MEDIA INTENSITY
    # ---------------------------------------------------------
    print("\n--- MEDIA COVERAGE ---")
    for col in ["numentions", "numsources", "numarticles", "avgtone"]:
        if col in df_zambia.columns:
            print(f"\n{col.upper()}")
            print(df_zambia[col].describe())

    # ---------------------------------------------------------
    # 5️⃣ ACTOR ANALYSIS
    # ---------------------------------------------------------
    print("\n--- TOP ACTOR1 COUNTRIES ---")
    if "actor1countrycode" in df_zambia.columns:
        print(df_zambia["actor1countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR2 COUNTRIES ---")
    if "actor2countrycode" in df_zambia.columns:
        print(df_zambia["actor2countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR1 NAMES ---")
    if "actor1name" in df_zambia.columns:
        print(df_zambia["actor1name"].value_counts().head(10))

    # ---------------------------------------------------------
    # 6️⃣ GEOGRAPHIC ANALYSIS (within Zambia)
    # ---------------------------------------------------------
    print("\n--- TOP ACTION GEO LOCATIONS ---")
    if "actiongeo_fullname" in df_zambia.columns:
        print(df_zambia["actiongeo_fullname"].value_counts().head(15))

    print("\n--- TOP ADM1 REGIONS ---")
    if "actiongeo_adm1code" in df_zambia.columns:
        print(df_zambia["actiongeo_adm1code"].value_counts().head(15))

    # ---------------------------------------------------------
    # 7️⃣ MISSINGNESS CHECK
    # ---------------------------------------------------------
    print("\n--- MISSINGNESS (% missing) ---")
    missing_pct = df_zambia.isna().mean() * 100
    print(missing_pct.sort_values(ascending=False).head(20))

    # ---------------------------------------------------------
    # 8️⃣ EXTREME EVENTS
    # ---------------------------------------------------------
    print("\n--- MOST NEGATIVE TONE EVENTS ---")
    if "avgtone" in df_zambia.columns:
        print(df_zambia.sort_values("avgtone").head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "avgtone"]
        ])

    print("\n--- MOST INTENSE (NUMARTICLES) ---")
    if "numarticles" in df_zambia.columns:
        print(df_zambia.sort_values("numarticles", ascending=False).head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "numarticles"]
        ])

    print("\nSummary complete.")



In [12]:
# there are duplicated urls with varying information. 
# this is because a single news article can generate multiple event rows.
# sometimes there are just duplicates
# sometimes there are multiple actors. 
# e.g first 4 rows refer to different actors in the same article. 

import json
import pandas as pd
import numpy as np

DROP_COLS = [
    "ingest_time",
    "globaleventid",
    "fractiondate",
    "isrootevent",
    "numentions",
    "numsources",
    "numarticles",
    "dateadded",
]

def _json_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    """Convert datetimes + NaNs to JSON-safe python types."""
    out = df.copy()

    # Convert datetime64 columns to ISO strings
    dt_cols = out.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
    for c in dt_cols:
        out[c] = out[c].dt.strftime("%Y-%m-%d")  # or .dt.isoformat if you prefer time too

    # Convert pandas Timestamps that may be in object columns
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].apply(lambda x: x.isoformat() if isinstance(x, pd.Timestamp) else x)

    # Replace NaN/NaT with empty string to stabilize dedupe + JSON
    out = out.replace({np.nan: "", pd.NaT: ""})

    return out

def collapse_by_sourceurl_store_variants(df: pd.DataFrame) -> pd.DataFrame:
    keep_df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore").copy()
    if "sourceurl" not in keep_df.columns:
        raise ValueError("Expected a 'sourceurl' column.")

    variant_cols = [c for c in keep_df.columns if c != "sourceurl"]

    # Make JSON-safe
    keep_df = _json_safe_df(keep_df)

    def build_variants(g: pd.DataFrame) -> str:
        unique_rows = g[variant_cols].drop_duplicates()
        variants = unique_rows.to_dict(orient="records")
        return json.dumps(variants, ensure_ascii=False, sort_keys=True)

    collapsed = (
        keep_df
        .groupby("sourceurl")
        .apply(lambda g: pd.Series({
            "n_variants": g[variant_cols].drop_duplicates().shape[0],
            "variants_json": build_variants(g),
        }))
        .reset_index()
    )

    return collapsed

In [13]:
df_collapsed = collapse_by_sourceurl_store_variants(df_zambia)
df_collapsed.to_csv("data/interim/gdelt_event_context_daily/2026/02/22/zambia_events_collapsed.csv", index=False)

These have been manually reviewed. For example, where a Zambia news site has been extracted, other articles on the same date have been checked and no others were published. Also a general web search for Zambia news has been completed and no other results have been found. This is not a guarantee that there are no other articles, but it is a good indication that the dataset is complete for this date.

I also researched the bigges news sites in Zambia and found that I was unable to open any articles from daily-mail.co.zm, which is one of the biggest news sites in Zambia owned by the government. I imagine this is because of geo-restrictions. 

I am now going to proceed with 3 steps:

1) Review 2 other dates to check for completeness and flag any further issues. This will be completed on the current analyse_raw.ipynb notebook. 

2) Manually find 10 articles, 5 mining-related and 5 non-mining related, from Zambia news and begin testing an LLM to extract any mention of mining.This will be completed on a subsequent notebook. 

3) Manually find 20 mining articles from Zambia news and begin testing an LLM to extract impacts from Mehrnoosh's long list. This will be completed on a subsequent notebook.

In [3]:
import pandas as pd 

df = pd.read_csv("data/interim/gdelt_event_context_daily/2025/02/22/20250222_events_full.csv")

df.head()

print("df shape:", df.shape)
print("non-null sourceurl:", df["sourceurl"].notna().sum())
print("sample sourceurl:", df["sourceurl"].dropna().head(3).tolist())
print("Total rows:", len(df))
print("Total columns:", df.shape[1])
na_counts = df.isna().sum()
na_percent = (na_counts / len(df)) * 100

na_summary = pd.DataFrame({
    "non_null_count": df.notna().sum(),
    "na_count": na_counts,
    "na_percent": na_percent.round(2)
}).sort_values("na_count", ascending=False)

na_summary

unique_counts = df.nunique().sort_values(ascending=False)
unique_counts

ZAMBIA_GEO_FIPS = "ZA"
ZAMBIA_ACTOR_CAMEO = "ZMB"

TEXT_TERMS = [
    "zambia", "zambian",
    "lusaka", "copperbelt", "ndola", "kitwe", "kabwe", "livingstone",
    "chingola", "solwezi", "chipata", "kasama", "mongu"
]

def contains_any(series, terms):
    s = series.fillna("").str.lower()
    mask = False
    for t in terms:
        mask = mask | s.str.contains(t, regex=False)
    return mask

geo_country_cols = ["actiongeo_countrycode", "actor1geo_countrycode", "actor2geo_countrycode"]
geo_adm1_cols    = ["actiongeo_adm1code", "actor1geo_adm1code", "actor2geo_adm1code"]
geo_full_cols    = ["actiongeo_fullname", "actor1geo_fullname", "actor2geo_fullname"]
actor_country_cols = ["actor1countrycode", "actor2countrycode"]
actor_name_cols = ["actor1name", "actor2name"]

geo_country_hit = False
for c in geo_country_cols:
    if c in df.columns:
        geo_country_hit = geo_country_hit | (df[c] == ZAMBIA_GEO_FIPS)

geo_adm1_hit = False
for c in geo_adm1_cols:
    if c in df.columns:
        geo_adm1_hit = geo_adm1_hit | df[c].fillna("").str.startswith(ZAMBIA_GEO_FIPS)

actor_country_hit = False
for c in actor_country_cols:
    if c in df.columns:
        actor_country_hit = actor_country_hit | (df[c] == ZAMBIA_ACTOR_CAMEO)

geo_text_hit = False
for c in geo_full_cols:
    if c in df.columns:
        geo_text_hit = geo_text_hit | contains_any(df[c], TEXT_TERMS)

actor_text_hit = False
for c in actor_name_cols:
    if c in df.columns:
        actor_text_hit = actor_text_hit | contains_any(df[c], ["zambia", "zambian"])

zambia_mask = geo_country_hit | geo_adm1_hit | actor_country_hit | geo_text_hit | actor_text_hit
df_zambia = df[zambia_mask].copy()

import pandas as pd
import numpy as np

def summarise_zambia(df_zambia: pd.DataFrame):
    print("="*80)
    print("ZAMBIA DATASET SUMMARY")
    print("="*80)

    # ---------------------------------------------------------
    # 1️⃣ BASIC INFO
    # ---------------------------------------------------------
    print("\n--- BASIC INFO ---")
    print(f"Total rows: {len(df_zambia):,}")
    print(f"Unique GlobalEventID: {df_zambia['globaleventid'].nunique():,}")

    if "sqldate" in df_zambia.columns:
        df_zambia["sqldate"] = pd.to_datetime(df_zambia["sqldate"], format="%Y%m%d", errors="coerce")
        print(f"Date range: {df_zambia['sqldate'].min()} → {df_zambia['sqldate'].max()}")


    # ---------------------------------------------------------
    # 3️⃣ EVENT TYPE DISTRIBUTION
    # ---------------------------------------------------------
    print("\n--- EVENT ROOT CODE DISTRIBUTION ---")
    if "eventrootcode" in df_zambia.columns:
        print(df_zambia["eventrootcode"].value_counts().head(15))

    print("\n--- QUADCLASS DISTRIBUTION ---")
    if "quadclass" in df_zambia.columns:
        print(df_zambia["quadclass"].value_counts())

    print("\n--- GOLDSTEIN SCALE SUMMARY ---")
    if "goldsteinscale" in df_zambia.columns:
        print(df_zambia["goldsteinscale"].describe())

    # ---------------------------------------------------------
    # 4️⃣ TONE & MEDIA INTENSITY
    # ---------------------------------------------------------
    print("\n--- MEDIA COVERAGE ---")
    for col in ["numentions", "numsources", "numarticles", "avgtone"]:
        if col in df_zambia.columns:
            print(f"\n{col.upper()}")
            print(df_zambia[col].describe())

    # ---------------------------------------------------------
    # 5️⃣ ACTOR ANALYSIS
    # ---------------------------------------------------------
    print("\n--- TOP ACTOR1 COUNTRIES ---")
    if "actor1countrycode" in df_zambia.columns:
        print(df_zambia["actor1countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR2 COUNTRIES ---")
    if "actor2countrycode" in df_zambia.columns:
        print(df_zambia["actor2countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR1 NAMES ---")
    if "actor1name" in df_zambia.columns:
        print(df_zambia["actor1name"].value_counts().head(10))

    # ---------------------------------------------------------
    # 6️⃣ GEOGRAPHIC ANALYSIS (within Zambia)
    # ---------------------------------------------------------
    print("\n--- TOP ACTION GEO LOCATIONS ---")
    if "actiongeo_fullname" in df_zambia.columns:
        print(df_zambia["actiongeo_fullname"].value_counts().head(15))

    print("\n--- TOP ADM1 REGIONS ---")
    if "actiongeo_adm1code" in df_zambia.columns:
        print(df_zambia["actiongeo_adm1code"].value_counts().head(15))

    # ---------------------------------------------------------
    # 7️⃣ MISSINGNESS CHECK
    # ---------------------------------------------------------
    print("\n--- MISSINGNESS (% missing) ---")
    missing_pct = df_zambia.isna().mean() * 100
    print(missing_pct.sort_values(ascending=False).head(20))

    # ---------------------------------------------------------
    # 8️⃣ EXTREME EVENTS
    # ---------------------------------------------------------
    print("\n--- MOST NEGATIVE TONE EVENTS ---")
    if "avgtone" in df_zambia.columns:
        print(df_zambia.sort_values("avgtone").head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "avgtone"]
        ])

    print("\n--- MOST INTENSE (NUMARTICLES) ---")
    if "numarticles" in df_zambia.columns:
        print(df_zambia.sort_values("numarticles", ascending=False).head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "numarticles"]
        ])

    print("\nSummary complete.")

# there are duplicated urls with varying information. 
# this is because a single news article can generate multiple event rows.
# sometimes there are just duplicates
# sometimes there are multiple actors. 
# e.g first 4 rows refer to different actors in the same article. 

import json
import pandas as pd
import numpy as np

DROP_COLS = [
    "ingest_time",
    "globaleventid",
    "fractiondate",
    "isrootevent",
    "numentions",
    "numsources",
    "numarticles",
    "dateadded",
]

EVENT_CODE_COLS = ["eventcode", "eventbasecode", "eventrootcode"]


def _json_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    """Convert datetimes + NaNs to JSON-safe python types."""
    out = df.copy()

    # Convert datetime64 columns to strings
    dt_cols = out.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
    for c in dt_cols:
        # keep just date if that's what sqldate represents
        out[c] = out[c].dt.strftime("%Y-%m-%d")

    # Convert any Timestamp objects that might live in object columns
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].apply(lambda x: x.isoformat() if isinstance(x, pd.Timestamp) else x)

    # Replace NaN / NaT with ""
    out = out.replace({np.nan: "", pd.NaT: ""})

    return out


def collapse_by_sourceurl_store_both_variants(df: pd.DataFrame) -> pd.DataFrame:
    """
    1 row per sourceurl.
    - variants_json: all unique row combinations across remaining columns (excluding sourceurl)
    - event_variants_json: all unique combinations across eventcode/eventbasecode/eventrootcode only
    """
    # 1) Drop requested columns (ignore missing)
    keep_df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore").copy()

    if "sourceurl" not in keep_df.columns:
        raise ValueError("Expected a 'sourceurl' column.")

    # 2) Ensure event code columns exist (if not, create empty to keep pipeline stable)
    for c in EVENT_CODE_COLS:
        if c not in keep_df.columns:
            keep_df[c] = ""

    # 3) Define columns that make up a "full variant"
    full_variant_cols = [c for c in keep_df.columns if c != "sourceurl"]

    # 4) Make JSON-safe
    keep_df = _json_safe_df(keep_df)

    def to_variants_json(g: pd.DataFrame, cols: list[str]) -> str:
        unique_rows = g[cols].drop_duplicates()
        records = unique_rows.to_dict(orient="records")
        return json.dumps(records, ensure_ascii=False, sort_keys=True)

    def agg(g: pd.DataFrame) -> pd.Series:
        full_unique = g[full_variant_cols].drop_duplicates()
        event_unique = g[EVENT_CODE_COLS].drop_duplicates()

        return pd.Series({
            "n_variants": len(full_unique),
            "variants_json": json.dumps(full_unique.to_dict(orient="records"), ensure_ascii=False, sort_keys=True),

            "n_event_variants": len(event_unique),
            "event_variants_json": json.dumps(event_unique.to_dict(orient="records"), ensure_ascii=False, sort_keys=True),
        })

    collapsed = (
        keep_df
        .groupby("sourceurl", sort=False)
        .apply(agg)
        .reset_index()
    )

    return collapsed




df_collapsed = collapse_by_sourceurl_store_both_variants(df_zambia)
df_collapsed.to_csv("data/interim/gdelt_event_context_daily/2025/02/22/zambia_events_collapsed.csv", index=False)

df shape: (91490, 62)
non-null sourceurl: 91490
sample sourceurl: ['https://www.ivpressonline.com/news/world/thousands-rally-in-slovakia-to-mark-the-2018-slayings-of-an-investigative-journalist-and-his/article_855c0b3e-d74b-5896-b765-ac3bb6e6f44a.html', 'https://www.dailytribune.com/2025/02/21/election-defenses-foreign-interference-cybersecurity/', 'https://www.dailytribune.com/2025/02/21/election-defenses-foreign-interference-cybersecurity/']
Total rows: 91490
Total columns: 62


In [2]:
import pandas as pd 

df = pd.read_csv("data/interim/gdelt_event_context_daily/2025/12/01/20251201_events_full.csv")

df.head()

print("df shape:", df.shape)
print("non-null sourceurl:", df["sourceurl"].notna().sum())
print("sample sourceurl:", df["sourceurl"].dropna().head(3).tolist())
print("Total rows:", len(df))
print("Total columns:", df.shape[1])
na_counts = df.isna().sum()
na_percent = (na_counts / len(df)) * 100

na_summary = pd.DataFrame({
    "non_null_count": df.notna().sum(),
    "na_count": na_counts,
    "na_percent": na_percent.round(2)
}).sort_values("na_count", ascending=False)

na_summary

unique_counts = df.nunique().sort_values(ascending=False)
unique_counts

ZAMBIA_GEO_FIPS = "ZA"
ZAMBIA_ACTOR_CAMEO = "ZMB"

TEXT_TERMS = [
    "zambia", "zambian",
    "lusaka", "copperbelt", "ndola", "kitwe", "kabwe", "livingstone",
    "chingola", "solwezi", "chipata", "kasama", "mongu"
]

def contains_any(series, terms):
    s = series.fillna("").str.lower()
    mask = False
    for t in terms:
        mask = mask | s.str.contains(t, regex=False)
    return mask

geo_country_cols = ["actiongeo_countrycode", "actor1geo_countrycode", "actor2geo_countrycode"]
geo_adm1_cols    = ["actiongeo_adm1code", "actor1geo_adm1code", "actor2geo_adm1code"]
geo_full_cols    = ["actiongeo_fullname", "actor1geo_fullname", "actor2geo_fullname"]
actor_country_cols = ["actor1countrycode", "actor2countrycode"]
actor_name_cols = ["actor1name", "actor2name"]

geo_country_hit = False
for c in geo_country_cols:
    if c in df.columns:
        geo_country_hit = geo_country_hit | (df[c] == ZAMBIA_GEO_FIPS)

geo_adm1_hit = False
for c in geo_adm1_cols:
    if c in df.columns:
        geo_adm1_hit = geo_adm1_hit | df[c].fillna("").str.startswith(ZAMBIA_GEO_FIPS)

actor_country_hit = False
for c in actor_country_cols:
    if c in df.columns:
        actor_country_hit = actor_country_hit | (df[c] == ZAMBIA_ACTOR_CAMEO)

geo_text_hit = False
for c in geo_full_cols:
    if c in df.columns:
        geo_text_hit = geo_text_hit | contains_any(df[c], TEXT_TERMS)

actor_text_hit = False
for c in actor_name_cols:
    if c in df.columns:
        actor_text_hit = actor_text_hit | contains_any(df[c], ["zambia", "zambian"])

zambia_mask = geo_country_hit | geo_adm1_hit | actor_country_hit | geo_text_hit | actor_text_hit
df_zambia = df[zambia_mask].copy()

import pandas as pd
import numpy as np

def summarise_zambia(df_zambia: pd.DataFrame):
    print("="*80)
    print("ZAMBIA DATASET SUMMARY")
    print("="*80)

    # ---------------------------------------------------------
    # 1️⃣ BASIC INFO
    # ---------------------------------------------------------
    print("\n--- BASIC INFO ---")
    print(f"Total rows: {len(df_zambia):,}")
    print(f"Unique GlobalEventID: {df_zambia['globaleventid'].nunique():,}")

    if "sqldate" in df_zambia.columns:
        df_zambia["sqldate"] = pd.to_datetime(df_zambia["sqldate"], format="%Y%m%d", errors="coerce")
        print(f"Date range: {df_zambia['sqldate'].min()} → {df_zambia['sqldate'].max()}")


    # ---------------------------------------------------------
    # 3️⃣ EVENT TYPE DISTRIBUTION
    # ---------------------------------------------------------
    print("\n--- EVENT ROOT CODE DISTRIBUTION ---")
    if "eventrootcode" in df_zambia.columns:
        print(df_zambia["eventrootcode"].value_counts().head(15))

    print("\n--- QUADCLASS DISTRIBUTION ---")
    if "quadclass" in df_zambia.columns:
        print(df_zambia["quadclass"].value_counts())

    print("\n--- GOLDSTEIN SCALE SUMMARY ---")
    if "goldsteinscale" in df_zambia.columns:
        print(df_zambia["goldsteinscale"].describe())

    # ---------------------------------------------------------
    # 4️⃣ TONE & MEDIA INTENSITY
    # ---------------------------------------------------------
    print("\n--- MEDIA COVERAGE ---")
    for col in ["numentions", "numsources", "numarticles", "avgtone"]:
        if col in df_zambia.columns:
            print(f"\n{col.upper()}")
            print(df_zambia[col].describe())

    # ---------------------------------------------------------
    # 5️⃣ ACTOR ANALYSIS
    # ---------------------------------------------------------
    print("\n--- TOP ACTOR1 COUNTRIES ---")
    if "actor1countrycode" in df_zambia.columns:
        print(df_zambia["actor1countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR2 COUNTRIES ---")
    if "actor2countrycode" in df_zambia.columns:
        print(df_zambia["actor2countrycode"].value_counts().head(10))

    print("\n--- TOP ACTOR1 NAMES ---")
    if "actor1name" in df_zambia.columns:
        print(df_zambia["actor1name"].value_counts().head(10))

    # ---------------------------------------------------------
    # 6️⃣ GEOGRAPHIC ANALYSIS (within Zambia)
    # ---------------------------------------------------------
    print("\n--- TOP ACTION GEO LOCATIONS ---")
    if "actiongeo_fullname" in df_zambia.columns:
        print(df_zambia["actiongeo_fullname"].value_counts().head(15))

    print("\n--- TOP ADM1 REGIONS ---")
    if "actiongeo_adm1code" in df_zambia.columns:
        print(df_zambia["actiongeo_adm1code"].value_counts().head(15))

    # ---------------------------------------------------------
    # 7️⃣ MISSINGNESS CHECK
    # ---------------------------------------------------------
    print("\n--- MISSINGNESS (% missing) ---")
    missing_pct = df_zambia.isna().mean() * 100
    print(missing_pct.sort_values(ascending=False).head(20))

    # ---------------------------------------------------------
    # 8️⃣ EXTREME EVENTS
    # ---------------------------------------------------------
    print("\n--- MOST NEGATIVE TONE EVENTS ---")
    if "avgtone" in df_zambia.columns:
        print(df_zambia.sort_values("avgtone").head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "avgtone"]
        ])

    print("\n--- MOST INTENSE (NUMARTICLES) ---")
    if "numarticles" in df_zambia.columns:
        print(df_zambia.sort_values("numarticles", ascending=False).head(5)[
            ["sqldate", "actor1name", "actor2name", "eventcode", "numarticles"]
        ])

    print("\nSummary complete.")

# there are duplicated urls with varying information. 
# this is because a single news article can generate multiple event rows.
# sometimes there are just duplicates
# sometimes there are multiple actors. 
# e.g first 4 rows refer to different actors in the same article. 

import json
import pandas as pd
import numpy as np

DROP_COLS = [
    "ingest_time",
    "globaleventid",
    "fractiondate",
    "isrootevent",
    "numentions",
    "numsources",
    "numarticles",
    "dateadded",
]

def _json_safe_df(df: pd.DataFrame) -> pd.DataFrame:
    """Convert datetimes + NaNs to JSON-safe python types."""
    out = df.copy()

    # Convert datetime64 columns to ISO strings
    dt_cols = out.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
    for c in dt_cols:
        out[c] = out[c].dt.strftime("%Y-%m-%d")  # or .dt.isoformat if you prefer time too

    # Convert pandas Timestamps that may be in object columns
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].apply(lambda x: x.isoformat() if isinstance(x, pd.Timestamp) else x)

    # Replace NaN/NaT with empty string to stabilize dedupe + JSON
    out = out.replace({np.nan: "", pd.NaT: ""})

    return out

def collapse_by_sourceurl_store_variants(df: pd.DataFrame) -> pd.DataFrame:
    keep_df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore").copy()
    if "sourceurl" not in keep_df.columns:
        raise ValueError("Expected a 'sourceurl' column.")

    variant_cols = [c for c in keep_df.columns if c != "sourceurl"]

    # Make JSON-safe
    keep_df = _json_safe_df(keep_df)

    def build_variants(g: pd.DataFrame) -> str:
        unique_rows = g[variant_cols].drop_duplicates()
        variants = unique_rows.to_dict(orient="records")
        return json.dumps(variants, ensure_ascii=False, sort_keys=True)

    collapsed = (
        keep_df
        .groupby("sourceurl")
        .apply(lambda g: pd.Series({
            "n_variants": g[variant_cols].drop_duplicates().shape[0],
            "variants_json": build_variants(g),
        }))
        .reset_index()
    )

    return collapsed

df_collapsed = collapse_by_sourceurl_store_variants(df_zambia)
df_collapsed.to_csv("data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed.csv", index=False)

df shape: (109383, 62)
non-null sourceurl: 109383
sample sourceurl: ['https://www.whas11.com/article/news/education/jcps-proposal-school-changes-2026-community-forums/417-dba17447-faf9-4699-8f01-0ea9567af681', 'https://www.chicagotribune.com/2025/11/30/trump-mri-results/', 'https://www.chicagotribune.com/2025/11/30/trump-mri-results/']
Total rows: 109383
Total columns: 62


2025 02 22 contained 12 articles. 
2025 12 01 contained 33 articles. 
2026 02 22 contained 7 articles. 

Very rough average of 17 articles per day. Across 10 years, this scales to around 62,000 articles which is unrealistic to use LLMs, costing ~£145 to analyse through LLMs. 
An approximate appropriate cost might be more like £20, suggesting we need to to reduce the number of articles by around 7x, to around 9,000 articles. 